# 03 - Full-grid SHAP values and the Fig. 5 map layers

**Run order:** `01_fit_full_model.py` -> **`03_shap_maps.ipynb`** -> `04_decision_plots.ipynb`
(`02_model_performance.py` is independent and can run any time after 01).

**Purpose.** Rebuild the 0.05-degree predictor table for every vegetated natural-land grid cell of
CONUS (the same mask that produced `data.csv`), z-score it, compute SHAP values of the fitted full
GPBoost model for every cell, and write the Fig. 5 map layers.

**Inputs** (paths relative to this folder; see `../data/README.md`)

| File | Content |
|---|---|
| `./models/GPBoost_GPP_LAI_full-model_ecoprovince.json` | booster fitted by `01_fit_full_model.py` |
| `../data/grid_005_layers/predictors/` | GOSIF-GPP (JJA 2017), GLOBMAP LAI, WorldClim bio10 / bio18, aridity index |
| `../data/grid_005_layers/PC_scores/` | functional composition PC1-PC3 |
| `../data/fd_layers_005/` | the eight functional diversity layers |
| `../data/grid_005_layers/masks/` | NLCD non-natural / non-vegetation fractions and majority class, MCD12C1 IGBP class, ecoprovince raster + lookup CSVs |

**Outputs**

| File | Content |
|---|---|
| `./results/scaler_dict.pkl` | per-variable `StandardScaler` fitted on the full valid grid |
| `./results/shap_values_original_scale.csv` | one row per valid cell: predictors in original units, SHAP values in GPP units (input to 04) |
| `./results/maps/fig5b_baseline_predicted_GPP.tif` | baseline prediction = mean GPP + SHAP(LAI) + SHAP(T) + SHAP(P) + SHAP(AI) |
| `./results/maps/fig5c_baseline_residual.tif` | observed GPP - baseline prediction |
| `./results/maps/fig5e_shap_sum_functional_composition.tif` | SHAP(PC1) + SHAP(PC2) + SHAP(PC3) |
| `./results/maps/fig5f_shap_sum_functional_diversity.tif` | SHAP(FRic_gamma) + SHAP(Fbeta_alpha_to_gamma) + SHAP(Fbeta_gamma_to_tau) + SHAP(FDiv_gamma) |
| `./results/maps/shap_sum_baseline_covariates.tif` | SHAP(LAI) + SHAP(T) + SHAP(P) + SHAP(AI) (intermediate) |

**Notes**
- All rasters share the same 1200 x 504 grid (pixel size 0.0485 degree, WGS84). Every layer is read
  into a flat array and stacked into one table, so no resampling happens here.
- The grid is z-scored with scalers fitted on the full valid grid, exactly as `data.csv` (a subsample
  of this grid) was built; the resulting scale/mean agree with those implied by `data.csv` to < 1 %.
- SHAP values come from `shap.Explainer` (TreeExplainer) on the boosted-tree part of the model; the
  spatial Gaussian process is not attributed. They are multiplied by the GPP standard deviation to
  express them in GPP units (g C m-2 d-1).
- Results depend on the booster from 01, which is tuned with an unseeded Optuna study, so values
  differ slightly between runs of 01.

In [ ]:
import os

import gpboost as gpb
import joblib
import numpy as np
import pandas as pd
import rasterio
import shap
from sklearn.preprocessing import StandardScaler

In [ ]:
DATA_DIR = '../data'
PRED_DIR = f'{DATA_DIR}/grid_005_layers/predictors'
PC_DIR = f'{DATA_DIR}/grid_005_layers/PC_scores'
FD_DIR = f'{DATA_DIR}/fd_layers_005'
MASK_DIR = f'{DATA_DIR}/grid_005_layers/masks'

MODEL_FILE = './models/GPBoost_GPP_LAI_full-model_ecoprovince.json'
RESULTS_DIR = './results'
MAP_DIR = f'{RESULTS_DIR}/maps'
os.makedirs(MAP_DIR, exist_ok=True)


def read_band(path, dtype=None):
    """Read band 1 of a raster as an array with NoData set to NaN."""
    dataset = rasterio.open(path)
    img = dataset.read(1)
    if dtype is not None:
        img = img.astype(dtype)
    img[img == dataset.nodata] = np.nan
    return dataset, img

# Read the 0.05-degree grid layers

In [ ]:
# GOSIF-GPP, June-August 2017 mean (stored x1000)
GPP_dataset, GPP_img = read_band(f'{PRED_DIR}/GPP_GOSIF_JJA2017.tif')
GPP_img = GPP_img / 1000

In [ ]:
# GLOBMAP LAI v3, growing-season 2017 mean (stored x100)
LAI_dataset, LAI_img = read_band(f'{PRED_DIR}/LAI_GLOBMAP_growing_season_2017.tif')
LAI_img = LAI_img / 100

In [ ]:
# functional composition: PC1-PC3 of the ten-trait PCA
trait_PCA_list = ['all_PC1', 'all_PC2', 'all_PC3']
trait_PCA_img_list = []
for pc in ['PC1', 'PC2', 'PC3']:
    _, img = read_band(f'{PC_DIR}/PCA_all_{pc}_005_clip.tif')
    trait_PCA_img_list.append(img)

In [ ]:
# NLCD 2019: fraction (%) of non-natural land cover (developed, cultivated, ...) per cell
non_natural_veg, non_natural_veg_img = read_band(f'{MASK_DIR}/NLCD_non_natural_fraction.tif')

In [ ]:
# NLCD 2019: fraction (%) of non-vegetated cover (water, barren, ice, developed, ...) per cell
non_veg, non_veg_img = read_band(f'{MASK_DIR}/NLCD_non_vegetation_fraction.tif')

In [ ]:
# WorldClim 2.1 bio10: mean temperature of the warmest quarter (degC)
temperature, temperature_img = read_band(f'{PRED_DIR}/Temperature_warmest_quarter_bio10.tif')

In [ ]:
# WorldClim 2.1 bio18: precipitation of the warmest quarter (mm)
precipitation, precipitation_img = read_band(f'{PRED_DIR}/Precipitation_warmest_quarter_bio18.tif')

In [ ]:
# the eight functional diversity layers used by the models (column name -> file)
FD_files = {
    'FRic_alpha': 'FRic_alpha.tif',
    'FDiv_alpha': 'FDiv_alpha.tif',
    'FRic_gamma': 'FRic_gamma.tif',
    'FDiv_gamma': 'FDiv_gamma.tif',
    'FRic_tau': 'FRic_tau.tif',
    'FDiv_tau': 'FDiv_tau.tif',
    'Fbeta_alpha_to_gamma': 'Fbeta_alpha-to-gamma.tif',
    'Fbeta_gamma_to_tau': 'Fbeta_gamma-to-tau.tif',
}
FD_all_list = list(FD_files)
FD_img_list = []
for FD, fname in FD_files.items():
    _, img = read_band(f'{FD_DIR}/{fname}')
    FD_img_list.append(img)

In [ ]:
# MODIS MCD12C1 IGBP land-cover class
IGBP, IGBP_img = read_band(f'{MASK_DIR}/MCD12C1_IGBP_class.tif', dtype=np.float32)

In [ ]:
# NLCD 2019 majority class per cell
nlcd_majority, nlcd_majority_img = read_band(f'{MASK_DIR}/NLCD_majority_class.tif', dtype=np.float32)

In [ ]:
# USDA Forest Service ecoprovince (integer raster value; decoded below)
ecoprovince, ecoprovince_img = read_band(f'{MASK_DIR}/ecoprovince.tif', dtype=np.float32)

## Latitude and longitude of every cell

In [ ]:
def get_lat_lon(dataset):
    lat = np.linspace(dataset.bounds.top, dataset.bounds.bottom, dataset.height)
    lon = np.linspace(dataset.bounds.left, dataset.bounds.right, dataset.width)
    return lat, lon


lat, lon = get_lat_lon(GPP_dataset)
lon_img, lat_img = np.meshgrid(lon, lat)

## Aridity index

In [ ]:
# Global Aridity Index v3 (annual), stored x10000
aridity_index, aridity_index_img = read_band(f'{PRED_DIR}/aridity_index_v3.tif', dtype=np.float32)
aridity_index_img = aridity_index_img / 10000

## Stack the layers into one table (one row per grid cell)

In [ ]:
data = np.stack(
    [GPP_img, LAI_img] + trait_PCA_img_list + [non_natural_veg_img, non_veg_img, temperature_img,
                                              precipitation_img] + FD_img_list + [
        IGBP_img, nlcd_majority_img, ecoprovince_img, lat_img, lon_img, aridity_index_img],
    axis=-1)
data = data.reshape(-1, data.shape[-1])

In [ ]:
data = pd.DataFrame(data, columns=['GPP', 'LAI'] + trait_PCA_list + ['NonNaturalVeg_ratio', 'NonVeg_ratio',
                                                                     'Temperature', 'Precipitation'] + FD_all_list + [
    'IGBP', 'nlcd_majority', 'ecoprovince', 'latitude', 'longitude', 'aridity_index'])

In [ ]:
# IGBP class codes -> names
IGBP_dict = {0: 'Water', 1: 'Evergreen Needleleaf forest', 2: 'Evergreen Broadleaf forest',
             3: 'Deciduous Needleleaf forest',
             4: 'Deciduous Broadleaf forest', 5: 'Mixed forest', 6: 'Closed shrubland', 7: 'Open shrubland',
             8: 'Woody savanna', 9: 'Savanna', 10: 'Grassland', 11: 'Permanent wetland', 12: 'Cropland',
             13: 'Urban and built-up',
             14: 'Cropland/Natural vegetation mosaic', 15: 'Snow and ice', 16: 'Barren or sparsely vegetated',
             17: 'Unclassified'}
data['IGBP'] = data['IGBP'].map(IGBP_dict)
IGBP_forest_list = ['Evergreen Needleleaf forest', 'Evergreen Broadleaf forest', 'Deciduous Needleleaf forest',
                    'Deciduous Broadleaf forest', 'Mixed forest']
data['IGBP_forest'] = data['IGBP'].apply(lambda x: 'Forest' if x in IGBP_forest_list else 'Non-forest')

In [ ]:
# NLCD majority class codes -> names (label only; the vegetated classes 41-95 are the ones used downstream)
nlcd_dict = {0: 'Open Water', 11: 'Developed, Open Space', 12: 'Developed, Low Intensity',
             21: 'Developed, Medium Intensity',
             22: 'Developed, High Intensity', 23: 'Developed, Open Space with Buildings',
             24: 'Developed, Open Space with Roads',
             31: 'Barren Land (Rock/Sand/Clay)', 41: 'Deciduous Forest', 42: 'Evergreen Forest', 43: 'Mixed Forest',
             51: 'Dwarf Scrub', 52: 'Shrub/Scrub', 71: 'Grassland/Herbaceous', 72: 'Sedge/Herbaceous', 73: 'Lichens',
             74: 'Moss', 81: 'Pasture/Hay', 82: 'Cultivated Crops', 90: 'Woody Wetlands',
             95: 'Emergent Herbaceous Wetlands'}
data['nlcd_majority'] = data['nlcd_majority'].map(nlcd_dict)

In [ ]:
# ecoprovince raster value -> province code (e.g. 'M221') -> province name
ecoprovince_value_dict = pd.read_csv(f'{MASK_DIR}/ecoprovince_raster_values.csv').set_index('Value')[
    'PROVINCE'].to_dict()
ecoprovince_name_dict = pd.read_csv(f'{MASK_DIR}/ecoprovince_code_and_name.csv').set_index('PROVINCE')[
    'PROVINCE_name'].to_dict()
data['ecoprovince'] = data['ecoprovince'].map(ecoprovince_value_dict)
data['ecoprovince_name'] = data['ecoprovince'].map(ecoprovince_name_dict)

In [ ]:
# UNEP aridity classes
data['aridity_index_classification'] = pd.cut(data['aridity_index'], bins=[0, 0.05, 0.2, 0.5, 0.65, 0.8, 100],
                                              labels=['Hyper-arid', 'Arid', 'Semi-arid', 'Dry sub-humid', 'Sub-humid',
                                                      'Humid'])

# Mask to vegetated natural land

A cell is kept when non-vegetated cover < 20 %, non-natural cover < 50 %, it is not hyper-arid, its IGBP
class is vegetated and non-agricultural, and it lies in a terrestrial ecoprovince. `valid_index` is the
boolean mask over the flattened grid; it is reused to write the maps.

In [ ]:
valid_index = (data['NonVeg_ratio'] < 20) & (data['NonNaturalVeg_ratio'] < 50) & (
        data['aridity_index_classification'] != 'Hyper-arid') & (
                  ~data['IGBP'].isin(
                      ['Water', 'Urban and built-up', 'Snow and ice', 'Barren or sparsely vegetated', 'Cropland',
                       'Cropland/Natural vegetation mosaic'])) & (
                      data['ecoprovince'] != 'Water')
print(valid_index.sum())

In [ ]:
# non-vegetated cover < 20 %
data = data[data['NonVeg_ratio'] < 20]
# non-natural cover < 50 %
data = data[data['NonNaturalVeg_ratio'] < 50]
# drop hyper-arid cells
data = data[data['aridity_index_classification'] != 'Hyper-arid']
# drop water bodies
data = data[data['ecoprovince'] != 'Water']
# IGBP filter
exclude_IGBP_list = ['Water', 'Urban and built-up', 'Snow and ice', 'Barren or sparsely vegetated',
                     'Cropland', 'Cropland/Natural vegetation mosaic']
data = data[~data['IGBP'].isin(exclude_IGBP_list)]
assert len(data) == valid_index.sum()

# Standardize the variables

Each variable is z-scored over the valid grid; the original values are kept in `<name>_original`
columns and the scalers are saved for the back-transformation of the SHAP values.

In [ ]:
scaler_dict = {}
columns_to_standardize = ['GPP', 'LAI'] + trait_PCA_list + FD_all_list + ['aridity_index'] + ['Temperature',
                                                                                             'Precipitation']
for column in columns_to_standardize:
    scaler = StandardScaler()
    data[column + '_original'] = data[column]
    data[column] = scaler.fit_transform(data[column].values.reshape(-1, 1)).flatten()
    scaler_dict[column] = scaler

joblib.dump(scaler_dict, f'{RESULTS_DIR}/scaler_dict.pkl')

# SHAP values of the full model

In [ ]:
baseline_var_list = ['LAI', 'Temperature', 'Precipitation', 'aridity_index']
full_model_var_list = ['all_PC1', 'all_PC2', 'all_PC3', 'FRic_gamma', 'Fbeta_alpha_to_gamma', 'Fbeta_gamma_to_tau',
                       'FDiv_gamma', ]
feature_list = baseline_var_list + full_model_var_list

In [ ]:
full_model = gpb.Booster(model_file=MODEL_FILE)
explainer = shap.Explainer(full_model)
shap_values = explainer(data[feature_list])

In [ ]:
# rescale SHAP values and base value from z-scored GPP to GPP units
explainer_original = shap.Explainer(full_model)
shap_values_original = explainer_original(data[feature_list])
shap_values_original.values = shap_values.values * scaler_dict['GPP'].scale_[0]
shap_values_original.base_values = shap_values.base_values * scaler_dict['GPP'].scale_[0] + scaler_dict['GPP'].mean_[0]

In [ ]:
# rescale the feature values to their original units
shap_data_list = []
for i, var in enumerate(feature_list):
    shap_data_list.append(shap_values.data[:, i] * scaler_dict[var].scale_[0] + scaler_dict[var].mean_[0])
shap_data = np.stack(shap_data_list, axis=-1)
shap_values_original.data = shap_data

In [ ]:
# one row per valid cell: features (original units), SHAP values (GPP units) and cell attributes
shap_values_original_df = pd.DataFrame(shap_values_original.data, columns=feature_list)

shap_values_original_df = pd.concat([shap_values_original_df,
                                     pd.DataFrame(shap_values_original.values,
                                                  columns=[x + '_SHAP' for x in feature_list])],
                                    axis=1)
shap_values_original_df.insert(0, 'GPP', data['GPP_original'].reset_index(drop=True))
shap_values_original_df.insert(2, 'ecoprovince', data['ecoprovince'].reset_index(drop=True))
shap_values_original_df.insert(3, 'IGBP', data['IGBP'].reset_index(drop=True))
shap_values_original_df.insert(4, 'aridity_index_classification',
                               data['aridity_index_classification'].reset_index(drop=True))
shap_values_original_df.insert(5, 'latitude', data['latitude'].reset_index(drop=True))
shap_values_original_df.insert(6, 'longitude', data['longitude'].reset_index(drop=True))
shap_values_original_df.insert(7, 'nlcd_majority', data['nlcd_majority'].reset_index(drop=True))
shap_values_original_df.to_csv(f'{RESULTS_DIR}/shap_values_original_scale.csv', index=False)

# Fig. 5 map layers

Every map is a flat array over the full grid (NaN outside `valid_index`) written back on the GPP grid.

In [ ]:
def write_map(values_flat, path):
    """Write a flattened full-grid array as a float32 GeoTIFF on the GPP grid."""
    with rasterio.open(path, 'w', driver='GTiff', width=GPP_dataset.width, height=GPP_dataset.height,
                       count=1, dtype=rasterio.float32,
                       crs=GPP_dataset.crs, transform=GPP_dataset.transform, nodata=np.nan) as dst:
        dst.write(values_flat.reshape(1, GPP_dataset.height, GPP_dataset.width).astype(rasterio.float32))

In [ ]:
# sum of the SHAP values of the baseline covariates (LAI, T, P, AI), GPP units
null_all_effect_sum = np.full(GPP_img.shape, np.nan).flatten()
null_all_effect_sum[valid_index] = shap_values[:, baseline_var_list].sum(axis=1).values * scaler_dict['GPP'].scale_[0]
write_map(null_all_effect_sum, f'{MAP_DIR}/shap_sum_baseline_covariates.tif')

In [ ]:
# Fig. 5c: residual of the baseline model = observed GPP - (mean GPP + baseline SHAP sum)
GPP_mean = np.full(GPP_img.shape, np.nan).flatten()
GPP_mean[valid_index] = np.full(shap_data.shape[0], scaler_dict['GPP'].mean_).flatten()
base_residual = GPP_img.flatten() - null_all_effect_sum - GPP_mean
write_map(base_residual, f'{MAP_DIR}/fig5c_baseline_residual.tif')

In [ ]:
# Fig. 5b: baseline prediction = mean GPP + baseline SHAP sum
base_predict = null_all_effect_sum + GPP_mean
write_map(base_predict, f'{MAP_DIR}/fig5b_baseline_predicted_GPP.tif')

In [ ]:
# Fig. 5e: summed SHAP of functional composition (PC1 + PC2 + PC3)
trait_effect_map = np.full(GPP_img.shape, np.nan).flatten()
trait_effect_map[valid_index] = shap_values[:, ['all_PC1', 'all_PC2', 'all_PC3', ]].sum(axis=1).values * \
                                scaler_dict['GPP'].scale_[0]
write_map(trait_effect_map, f'{MAP_DIR}/fig5e_shap_sum_functional_composition.tif')

In [ ]:
# Fig. 5f: summed SHAP of functional diversity (FRic_gamma + Fbeta_alpha_to_gamma + Fbeta_gamma_to_tau + FDiv_gamma)
FD_all_effect_sum = np.full(GPP_img.shape, np.nan).flatten()
FD_all_effect_sum[valid_index] = shap_values[:, ['FRic_gamma', 'Fbeta_alpha_to_gamma', 'Fbeta_gamma_to_tau',
                                                 'FDiv_gamma']].sum(axis=1).values * scaler_dict['GPP'].scale_[0]
write_map(FD_all_effect_sum, f'{MAP_DIR}/fig5f_shap_sum_functional_diversity.tif')